# Lesson 3: AI Pair Coding Workflow

## Objective

Model how Agent A and Agent B can pair-code across different tools while sharing one Geond evidence layer.

## Prerequisites

- Complete Lesson 1 or run `uv run geond seed-sample` first.
- Understand that Geond is the shared substrate, not the agent runner.
- Optional: read `docs/antigravity_codex_geond_verification.md` for one concrete verified example.

## Safety

This lesson uses Agent A and Agent B labels by default. Codex + Antigravity is mentioned as a verified example only; do not import private transcripts unless you explicitly opt in.


In [ ]:
import json
import subprocess
from pathlib import Path

REPO = Path.cwd()
WORKSPACE_URI = "file:///sample/geond"


def run(args, check=True):
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=REPO, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result


workspace_id = json.loads(run(["uv", "run", "geond", "seed-sample"]).stdout)["workspace_id"]

## Run: Agent A reads context

Expected outcome: Agent A gets a compact review of prior context and current coordination state before editing.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "review-context",
        WORKSPACE_URI,
        "--intent",
        "Agent A plans a small service change",
        "--symbol",
        "build_answer",
        "--format",
        "markdown",
    ]
)

## Run: Agent B records work

Expected outcome: Agent B leaves an action record and a changeset-shaped evidence item for the same workspace.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "record-agent-action",
        workspace_id,
        "--agent-name",
        "agent-b",
        "--action-kind",
        "task_start",
        "--summary",
        "Agent B is checking build_answer after Agent A review.",
    ]
)
run(
    [
        "uv",
        "run",
        "geond",
        "record-changeset",
        "--workspace-uri",
        WORKSPACE_URI,
        "--workspace-name",
        "geond-sample",
        "--agent-name",
        "agent-b",
        "--summary",
        "Tutorial changeset: no real files were modified.",
        "--file",
        "examples/python_service/service.py",
    ]
)

## Run: both agents share reservations and handoffs

Expected outcome: the pair-coding workflow has explicit ownership and a next-action packet.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "reserve-files",
        workspace_id,
        "--agent-name",
        "agent-a",
        "--file",
        "examples/python_service/service.py",
        "--purpose",
        "Pair-coding tutorial ownership signal",
    ]
)
run(
    [
        "uv",
        "run",
        "geond",
        "record-handoff",
        workspace_id,
        "--from-agent",
        "agent-a",
        "--to-agent",
        "agent-b",
        "--summary",
        "Agent A reviewed context; Agent B should verify before editing.",
        "--next-action",
        "Inspect dashboard events and code risk before continuing.",
    ]
)

## Run: reviewer sees one trail

Expected outcome: dashboard read models expose the pair activity without reading every raw transcript.


In [ ]:
run(["uv", "run", "geond", "dashboard-overview", workspace_id, "--limit", "10"])
run(["uv", "run", "geond", "dashboard-events", workspace_id, "--limit", "20"])

## Concrete example: Codex + Antigravity

The same pair-coding pattern has been verified locally with Codex and Antigravity. Keep that as an example, not the product boundary: Copilot, Claude Code, Continue, Manus, or custom MCP agents can use the same Geond shared evidence model.

## Cleanup

Purge sample state when done:

```bash
uv run geond purge-workspace file:///sample/geond --yes
```
